In [1]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
import torch
print(f"Torch version: {torch.__version__}")
print ("Is GPU avaliable?: ", torch.cuda.is_available())
print ("GPU name: ", torch.cuda.get_device_name())

Torch version: 2.5.1+cu121
Is GPU avaliable?:  True
GPU name:  NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
np.random.seed(2026)

In [4]:
positive_ex = list(open('data/rt-polarity.pos', 'r', encoding = 'utf-8').readlines())
positive_ex = [s.strip() for s in positive_ex]
negative_ex = list(open('data/rt-polarity.neg', 'r', encoding = 'utf-8').readlines())
negative_ex = [s.strip() for s in negative_ex]

In [5]:
df = pd.DataFrame({'text': negative_ex + positive_ex,
                   'lable': [0]*len(negative_ex) + [1]*len(positive_ex)})
df = df.reset_index(drop=True)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['lable'], stratify=df['lable'])

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Current hardware is {device.upper()}')

model = SentenceTransformer('all-MiniLM-L6-v2', device = device)

train_embeddings = model.encode(X_train.to_list(), batch_size = 128, normalize_embeddings = True, show_progress_bar = True)
test_embeddings = model.encode(X_test.to_list(), batch_size = 128, normalize_embeddings = True, show_progress_bar = True)

Current hardware is CUDA


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [8]:
print(len(X_train), len(train_embeddings))

7996 7996


In [9]:
d = train_embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(np.array(train_embeddings).astype('float32'))
print(f"Number of vectors in the index: {index.ntotal}")

Number of vectors in the index: 7996


In [10]:
k = 5

distances, indices = index.search(np.array(test_embeddings).astype('float32'), k)

In [11]:
predictions = []
for neighbor_idxs in indices:
    neighbor_labels = [y_train.tolist()[i] for i in neighbor_idxs]
    pred = max(set(neighbor_labels), key=neighbor_labels.count)
    predictions.append(pred)

In [12]:
metrics_df = pd.DataFrame(columns=['Method', 'Accuracy', 'Precision', 'Recall', 'F1'])

In [13]:
acc = accuracy_score(y_test, predictions)
prec = precision_score(y_test, predictions, average='binary')
rec = recall_score(y_test, predictions, average='binary')
f1 = f1_score(y_test, predictions, average='binary')

new_row = pd.DataFrame([{
    'Method': 'FAISS',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1
}])

metrics_df = pd.concat([metrics_df, new_row], ignore_index=True)

display(metrics_df)

,Method,Accuracy,Precision,Recall,F1
0,FAISS,0.685671,0.660819,0.762941,0.708217


In [14]:
from transformers import pipeline
import gc

gc.collect()
torch.cuda.empty_cache()

In [15]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module = "torch.utils.data")

In [ ]:
zero_shot_classifier = pipeline("zero-shot-classification", model = "cross-encoder/nli-distilroberta-base", device_map= 'auto', dtype=torch.float16)

labels = ["positive", "negative"]
results = zero_shot_classifier(X_test.to_list(), labels, batch_size = 16, multi_label = False)

pred = [1 if res["labels"][0] == "positive" else 0 for res in results]

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

new_row = pd.DataFrame([{
    'Method': 'Zero-shot',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1
}])

metrics_df = pd.concat([metrics_df, new_row], ignore_index = True)
display(metrics_df)

for review, res in zip(X_test.iloc[:5], results[:5]):
    print("-"*50)
    print(f"Review: {review[:100]}...")
    print(f"Predicted: {res['labels'][0]}, confidence - {res['scores'][0]:.4f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,Method,Accuracy,Precision,Recall,F1
0,FAISS,0.685671,0.660819,0.762941,0.708217
1,Zero-shot,0.744561,0.825349,0.620405,0.708351


--------------------------------------------------
Review: more intellectually scary than dramatically involving ....
Predicted: negative, confidence - 0.9311
--------------------------------------------------
Review: the sinister inspiration that fuelled devito's early work is confused in death to smoochy into somet...
Predicted: negative, confidence - 0.9939
--------------------------------------------------
Review: this is a dark , gritty , sometimes funny little gem ....
Predicted: negative, confidence - 0.8587
--------------------------------------------------
Review: a hugely rewarding experience that's every bit as enlightening , insightful and entertaining as gran...
Predicted: positive, confidence - 0.9978
--------------------------------------------------
Review: so faithful to the doldrums of the not-quite-urban , not-quite-suburban milieu as to have viewers re...
Predicted: negative, confidence - 0.8130


In [17]:
tfidf_vectorizer = TfidfVectorizer(min_df=5)
X_train_tfidf_vectorized = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf_vectorized = tfidf_vectorizer.transform(X_test)

print('Number of features = {:,}'.format(len(tfidf_vectorizer.get_feature_names_out())))
print('Shape of X_train_vectorized:', X_train_tfidf_vectorized.shape)

Number of features = 3,686
Shape of X_train_vectorized: (7996, 3686)


In [18]:
sorted_tfidf_index = X_train_tfidf_vectorized.max(axis=0).toarray()[0].argsort()
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

print('Smallest tfidf:\n', feature_names[sorted_tfidf_index[:10]])
print('Largest tfidf:\n', feature_names[sorted_tfidf_index[-10:]])

Smallest tfidf:
 ['ponder' 'planet' 'specific' 'ham' 'lavish' 'ten' 'jim' 'cia' 'surprised'
 'benigni']
Largest tfidf:
 ['refreshing' 'reality' 'uneven' 'calculated' 'fantastic' 'experience'
 'joyous' 'indeed' 'stupid' 'film']


In [19]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf_vectorized, y_train)

predictions = clf.predict(X_test_tfidf_vectorized)

In [20]:
acc = accuracy_score(y_test, predictions)
prec = precision_score(y_test, predictions)
rec = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

new_row = pd.DataFrame([{
    'Method': 'TF-IDF + Logistic Regression',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1
}])

metrics_df = pd.concat([metrics_df, new_row], ignore_index=True)

display(metrics_df)

,Method,Accuracy,Precision,Recall,F1
0,FAISS,0.685671,0.660819,0.762941,0.708217
1,Zero-shot,0.744561,0.825349,0.620405,0.708351
2,TF-IDF + Logistic Regression,0.753188,0.754333,0.750938,0.752632


The TF-IDF with Logistic Regression showed the best and most stable results, achieving the best overall balance of precision and recall

Zero-shot model was highly precise, but struggling with recall. This method was also the slowest one. I tried loading larger model (Facebook Bart), but it was too large for my pc to handle. , i think that larger models can achieve better results.

FAISS had the lowest accuracy. Its quite fast, so it can be used for a quick search for a texts similarity, but rather not for a maximum classification accuracy